# 04C — Final Test Candidate Generation — LoRA Locked — FINAL V1 CACHE-SAFE

Notebook ini **hanya melakukan inference final** untuk skenario **Fine-Tuning LoRA** menggunakan adapter yang sudah dibangun kembali oleh `03B_lora_locked_retrain_FINAL_V2_CACHE_SAFE`.

## Protokol final

- Backbone: `meta-llama/Llama-3.1-8B-Instruct`
- Adapter: `Artifacts/lora_locked_adapter/`
- LoRA locked: **L3, r=16, alpha=32, LR=2e-4, dropout=0.00**
- Checkpoint locked: **checkpoint-159 / epoch 3**
- Test: **53 input**
- Kandidat per input: **5**
- Seed: `42, 43, 44, 45, 46`
- `temperature=0.70`
- `top_p=0.95`
- `top_k=0`
- `num_beams=1`
- `max_new_tokens=128`
- BF16, tanpa kuantisasi base model
- Reference Indonesia test **tidak dibaca sama sekali**
- Belum menghitung S3E, MBR, BLEU, chrF++, atau LLM evaluator

## Cache-safe

Backbone/tokenizer hanya boleh dibaca dari cache lokal:

`/workspace/.hf_home/hub`

Notebook memakai `local_files_only=True`. Jika cache tidak tersedia, notebook **berhenti**, bukan mengunduh ulang model.

## Output utama

`Hasil/final_generation/lora_locked/run_<id>/candidates_long.csv`

Notebook mendukung **resume per kandidat**. Jika terputus, jalankan kembali notebook; kandidat yang sudah tersimpan tidak dibangkitkan ulang.

## 1. Dependency audit — tanpa `pip install`

In [1]:
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

EXPECTED_EXACT = {
    "transformers": "4.48.1",
    "accelerate": "1.2.1",
    "huggingface_hub": "0.27.1",
    "peft": "0.14.0",
}

errors = []

for package_name, expected in EXPECTED_EXACT.items():
    try:
        actual = version(package_name)
    except PackageNotFoundError:
        errors.append(f"{package_name}: NOT INSTALLED")
        continue

    ok = Version(actual) == Version(expected)
    print(f"{package_name:18} {actual:12} {'OK' if ok else 'MISMATCH'}")

    if not ok:
        errors.append(
            f"{package_name}: actual={actual}, expected={expected}"
        )

if errors:
    raise RuntimeError(
        "Dependency audit gagal:\n- "
        + "\n- ".join(errors)
        + "\nGunakan environment yang sama dengan 03B V2."
    )

print("\nDEPENDENCY AUDIT: PASS")

transformers       4.48.1       OK
accelerate         1.2.1        OK
huggingface_hub    0.27.1       OK
peft               0.14.0       OK

DEPENDENCY AUDIT: PASS


## 2. Environment, GPU, cache, config lock, dan adapter audit

In [2]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import platform
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import get_token
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
)

PROJECT_ROOT = Path(
    "/workspace/10122233_CikitaMeydilvira_Skripsi"
).resolve()

HF_HOME = Path(
    "/workspace/.hf_home"
).resolve()

HF_HUB_CACHE = (
    HF_HOME / "hub"
).resolve()

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_CACHE_DIR = (
    HF_HUB_CACHE
    / "models--meta-llama--Llama-3.1-8B-Instruct"
)

CONFIG_PATH = (
    PROJECT_ROOT
    / "Config"
    / "lora_locked_config.json"
)

ADAPTER_DIR = (
    PROJECT_ROOT
    / "Artifacts"
    / "lora_locked_adapter"
)

TEST_PATH = (
    PROJECT_ROOT
    / "Dataset"
    / "test.csv"
)

OUTPUT_BASE = (
    PROJECT_ROOT
    / "Hasil"
    / "final_generation"
    / "lora_locked"
)

DEVICE_INDEX = 0
DTYPE = torch.bfloat16

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA tidak tersedia.")

if not torch.cuda.is_bf16_supported():
    raise RuntimeError("GPU tidak mendukung BF16.")

free_b, total_b = torch.cuda.mem_get_info(DEVICE_INDEX)
free_gib = free_b / (1024 ** 3)
total_gib = total_b / (1024 ** 3)

print(
    f"GPU preflight : {free_gib:.2f} GiB free / "
    f"{total_gib:.2f} GiB total"
)

if free_gib < 20.0:
    raise RuntimeError(
        f"VRAM bebas hanya {free_gib:.2f} GiB. "
        "Shutdown kernel/proses GPU lain sebelum 04C."
    )

if not MODEL_CACHE_DIR.exists():
    raise FileNotFoundError(
        f"Cache Llama tidak ditemukan: {MODEL_CACHE_DIR}. "
        "04C sengaja tidak mengunduh ulang model."
    )

cache_size_bytes = sum(
    p.stat().st_size
    for p in MODEL_CACHE_DIR.rglob("*")
    if p.is_file()
)
cache_size_gib = cache_size_bytes / (1024 ** 3)

if cache_size_gib < 10.0:
    raise RuntimeError(
        f"Cache Llama tampak tidak lengkap: {cache_size_gib:.2f} GiB."
    )

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"LoRA locked config tidak ditemukan: {CONFIG_PATH}. "
        "Jalankan 03B V2 sampai COMPLETE."
    )

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(
        f"Adapter final tidak ditemukan: {ADAPTER_DIR}. "
        "Jalankan 03B V2 sampai COMPLETE."
    )

if not (ADAPTER_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(
        f"adapter_config.json tidak ditemukan di {ADAPTER_DIR}"
    )

if not (
    (ADAPTER_DIR / "adapter_model.safetensors").exists()
    or (ADAPTER_DIR / "adapter_model.bin").exists()
):
    raise FileNotFoundError(
        f"Bobot adapter tidak ditemukan di {ADAPTER_DIR}"
    )

cfg = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

EXPECTED_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

config_checks = {
    "model_id":
        cfg.get("model_id") == MODEL_ID,
    "r":
        int(cfg.get("r", cfg.get("rank", -1))) == 16,
    "alpha":
        int(cfg.get("alpha", -1)) == 32,
    "dropout":
        np.isclose(float(cfg.get("dropout", -1)), 0.0),
    "learning_rate":
        np.isclose(float(cfg.get("learning_rate", -1)), 2e-4),
    "target_modules":
        list(cfg.get("target_modules", [])) == EXPECTED_TARGET_MODULES,
    "checkpoint":
        cfg.get("selected_checkpoint_name") == "checkpoint-159",
    "epoch":
        np.isclose(float(cfg.get("selected_checkpoint_epoch", -1)), 3.0),
    "max_sequence_length":
        int(cfg.get("max_sequence_length", -1)) == 640,
    "test_read_false":
        cfg.get("test_read") is False,
}

failed = [
    name
    for name, ok in config_checks.items()
    if not bool(ok)
]

if failed:
    raise RuntimeError(
        f"LoRA locked config audit gagal: {failed}"
    )

adapter_cfg = json.loads(
    (ADAPTER_DIR / "adapter_config.json").read_text(
        encoding="utf-8"
    )
)

adapter_targets = adapter_cfg.get("target_modules", [])
if isinstance(adapter_targets, set):
    adapter_targets = list(adapter_targets)

adapter_checks = {
    "r":
        int(adapter_cfg.get("r", -1)) == 16,
    "alpha":
        int(adapter_cfg.get("lora_alpha", -1)) == 32,
    "dropout":
        np.isclose(
            float(adapter_cfg.get("lora_dropout", -1)),
            0.0,
        ),
    "target_modules":
        set(adapter_targets) == set(EXPECTED_TARGET_MODULES),
}

adapter_failed = [
    name
    for name, ok in adapter_checks.items()
    if not bool(ok)
]

if adapter_failed:
    raise RuntimeError(
        f"Adapter config audit gagal: {adapter_failed}"
    )

print("PROJECT_ROOT    :", PROJECT_ROOT)
print("HF_HUB_CACHE    :", HF_HUB_CACHE)
print("Llama cache GiB :", f"{cache_size_gib:.2f}")
print("CONFIG          :", CONFIG_PATH)
print("ADAPTER         :", ADAPTER_DIR)
print("Checkpoint lock :", cfg["selected_checkpoint_name"])
print("Validation chrF++:", cfg.get("validation_chrfpp"))
print("Validation BLEU :", cfg.get("validation_bleu"))
print("GPU             :", torch.cuda.get_device_name(DEVICE_INDEX))
print("BF16            :", torch.cuda.is_bf16_supported())
print("HF token cached :", bool(get_token()))

print("\nLOCK + CACHE + GPU AUDIT: PASS")

GPU preflight : 22.48 GiB free / 23.52 GiB total
PROJECT_ROOT    : /workspace/10122233_CikitaMeydilvira_Skripsi
HF_HUB_CACHE    : /workspace/.hf_home/hub
Llama cache GiB : 29.93
CONFIG          : /workspace/10122233_CikitaMeydilvira_Skripsi/Config/lora_locked_config.json
ADAPTER         : /workspace/10122233_CikitaMeydilvira_Skripsi/Artifacts/lora_locked_adapter
Checkpoint lock : checkpoint-159
Validation chrF++: 34.78665232664383
Validation BLEU : 11.057929303516635
GPU             : NVIDIA GeForce RTX 4090
BF16            : True
HF token cached : True

LOCK + CACHE + GPU AUDIT: PASS


## 3. Test input audit — reference tidak dibaca

In [3]:
TEST_COLUMNS = [
    "row_id",
    "conversation_no",
    "turn_order",
    "duri_text",
]

if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Test dataset tidak ditemukan: {TEST_PATH}"
    )

# PENTING:
# indonesia_text tidak pernah dibaca ke memory pada notebook 04C.
test_df = pd.read_csv(
    TEST_PATH,
    usecols=TEST_COLUMNS,
)

if len(test_df) != 53:
    raise RuntimeError(
        f"Test harus 53 input, ditemukan {len(test_df)}."
    )

if test_df["row_id"].duplicated().any():
    dup = test_df.loc[
        test_df["row_id"].duplicated(keep=False),
        "row_id",
    ].tolist()
    raise RuntimeError(
        f"row_id test duplikat: {dup[:10]}"
    )

if test_df["duri_text"].isna().any():
    raise RuntimeError("Ada duri_text test yang NaN.")

if (
    test_df["duri_text"]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):
    raise RuntimeError("Ada duri_text test yang kosong.")

if "indonesia_text" in test_df.columns:
    raise RuntimeError(
        "REFERENCE LEAKAGE: indonesia_text masuk ke memory."
    )

source_payload = "\n".join(
    f"{r.row_id}\t{r.conversation_no}\t{r.turn_order}\t{r.duri_text}"
    for r in test_df.itertuples(index=False)
).encode("utf-8")

TEST_SOURCE_FINGERPRINT = hashlib.sha256(
    source_payload
).hexdigest()

print("Test rows              :", len(test_df))
print("Test conversations     :", test_df["conversation_no"].nunique())
print("Columns in memory      :", test_df.columns.tolist())
print("Reference Indonesia    : TIDAK DIBACA")
print("Test source fingerprint:", TEST_SOURCE_FINGERPRINT)
print("\nTEST AUDIT: PASS")

Test rows              : 53
Test conversations     : 3
Columns in memory      : ['row_id', 'conversation_no', 'turn_order', 'duri_text']
Reference Indonesia    : TIDAK DIBACA
Test source fingerprint: 5c8cef6129918556b9d28b651d345bf5111e2dcc0dc11ce94605b2a2656b2b1f

TEST AUDIT: PASS


## 4. Prompt dan konfigurasi generasi final

In [4]:
SYSTEM_MESSAGE = (
    "Anda adalah penerjemah bahasa Duri ke bahasa Indonesia."
)

USER_TEMPLATE = (
    "Terjemahkan teks bahasa Duri berikut ke bahasa Indonesia.\n"
    "Keluarkan hanya hasil terjemahan tanpa penjelasan tambahan.\n\n"
    "Bahasa Duri: {duri_text}\n"
    "Bahasa Indonesia:"
)

SEEDS = [42, 43, 44, 45, 46]

GENERATION_CONFIG = {
    "temperature": 0.70,
    "top_p": 0.95,
    "top_k": 0,
    "num_beams": 1,
    "max_new_tokens": 128,
}

PROMPT_PROTOCOL = "llama_chat_system_plus_user_translation_v6"

prompt_signature_raw = json.dumps(
    {
        "system": SYSTEM_MESSAGE,
        "user_template": USER_TEMPLATE,
        "protocol": PROMPT_PROTOCOL,
    },
    ensure_ascii=False,
    sort_keys=True,
).encode("utf-8")

PROMPT_SIGNATURE = hashlib.sha256(
    prompt_signature_raw
).hexdigest()

def build_user_messages(
    duri_text: str,
) -> list[dict[str, str]]:
    return [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE,
        },
        {
            "role": "user",
            "content": USER_TEMPLATE.format(
                duri_text=duri_text
            ),
        },
    ]

def clean_translation_output(
    text: str,
) -> str:
    value = str(text).strip()

    prefixes = [
        "Bahasa Indonesia:",
        "Terjemahan:",
        "Jawaban:",
    ]

    changed = True
    while changed:
        changed = False
        for prefix in prefixes:
            if value.lower().startswith(
                prefix.lower()
            ):
                value = value[len(prefix):].strip()
                changed = True

    return value.strip()

print("Seeds           :", SEEDS)
print("Generation      :", GENERATION_CONFIG)
print("Prompt protocol :", PROMPT_PROTOCOL)
print("Prompt signature:", PROMPT_SIGNATURE)

Seeds           : [42, 43, 44, 45, 46]
Generation      : {'temperature': 0.7, 'top_p': 0.95, 'top_k': 0, 'num_beams': 1, 'max_new_tokens': 128}
Prompt protocol : llama_chat_system_plus_user_translation_v6
Prompt signature: 9e36da98076b91c2a9941d93efebb3da94cdadafd3236608f42092547e73c7ea


## 5. Load tokenizer + base Llama + adapter LoRA dari cache lokal

In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=get_token(),
    use_fast=True,
    trust_remote_code=False,
    cache_dir=str(HF_HUB_CACHE),
    local_files_only=True,
)

if tokenizer.chat_template is None:
    raise RuntimeError(
        "Tokenizer tidak memiliki chat_template."
    )

if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is None:
        raise RuntimeError(
            "Tokenizer tidak memiliki pad/eos token."
        )
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Loading BASE model from LOCAL cache only...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=get_token(),
    torch_dtype=DTYPE,
    device_map={"": DEVICE_INDEX},
    low_cpu_mem_usage=True,
    trust_remote_code=False,
    cache_dir=str(HF_HUB_CACHE),
    local_files_only=True,
)

if (
    bool(getattr(base_model, "is_loaded_in_4bit", False))
    or bool(getattr(base_model, "is_loaded_in_8bit", False))
):
    raise RuntimeError(
        "Base model terkuantisasi. Final LoRA harus BF16, bukan QLoRA."
    )

print("Loading LoRA adapter from LOCAL artifact...")

model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_DIR),
    is_trainable=False,
)

model.eval()
model.config.use_cache = True

try:
    model.gradient_checkpointing_disable()
except Exception:
    pass

terminator_ids = []

if tokenizer.eos_token_id is not None:
    terminator_ids.append(
        int(tokenizer.eos_token_id)
    )

eot = tokenizer.convert_tokens_to_ids(
    "<|eot_id|>"
)

if (
    isinstance(eot, int)
    and eot >= 0
    and eot != tokenizer.unk_token_id
    and eot not in terminator_ids
):
    terminator_ids.append(eot)

free_b, total_b = torch.cuda.mem_get_info(DEVICE_INDEX)

print("\nMODEL + ADAPTER LOADED")
print(
    f"Free VRAM: {free_b/(1024**3):.2f} GiB / "
    f"{total_b/(1024**3):.2f} GiB"
)
print("Terminator IDs:", terminator_ids)

Loading BASE model from LOCAL cache only...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading LoRA adapter from LOCAL artifact...

MODEL + ADAPTER LOADED
Free VRAM: 7.02 GiB / 23.52 GiB
Terminator IDs: [128009]


## 6. Generation helper

In [6]:
def generate_one_candidate(
    duri_text: str,
    seed: int,
) -> tuple[str, float]:

    set_seed(int(seed))

    messages = build_user_messages(
        duri_text
    )

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    encoded = {
        key: value.to(model.device)
        for key, value in encoded.items()
    }

    kwargs = {
        "max_new_tokens":
            GENERATION_CONFIG["max_new_tokens"],
        "do_sample":
            True,
        "temperature":
            GENERATION_CONFIG["temperature"],
        "top_p":
            GENERATION_CONFIG["top_p"],
        "top_k":
            GENERATION_CONFIG["top_k"],
        "num_beams":
            GENERATION_CONFIG["num_beams"],
        "repetition_penalty":
            1.0,
        "use_cache":
            True,
        "pad_token_id":
            tokenizer.pad_token_id,
    }

    if terminator_ids:
        kwargs["eos_token_id"] = terminator_ids

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **encoded,
            **kwargs,
        )

    elapsed = time.perf_counter() - start

    prompt_tokens = int(
        encoded["input_ids"].shape[-1]
    )

    generated_ids = outputs[
        0,
        prompt_tokens:
    ]

    decoded = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    return (
        clean_translation_output(decoded),
        float(elapsed),
    )

print("Generation helper: READY")

Generation helper: READY


## 7. Output run + resume audit

In [7]:
OUTPUT_BASE.mkdir(
    parents=True,
    exist_ok=True,
)

ACTIVE_RUN_PATH = (
    OUTPUT_BASE
    / "active_run.json"
)

if ACTIVE_RUN_PATH.exists():
    active = json.loads(
        ACTIVE_RUN_PATH.read_text(
            encoding="utf-8"
        )
    )
    RUN_ID = str(active["run_id"])
else:
    RUN_ID = (
        "run_"
        + datetime.now(timezone.utc).strftime(
            "%Y%m%dT%H%M%SZ"
        )
    )
    active = {
        "run_id": RUN_ID,
        "created_at_utc":
            datetime.now(timezone.utc).isoformat(),
        "status": "running",
    }
    ACTIVE_RUN_PATH.write_text(
        json.dumps(
            active,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

RUN_DIR = OUTPUT_BASE / RUN_ID
RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LONG_PATH = RUN_DIR / "candidates_long.csv"
WIDE_PATH = RUN_DIR / "candidates_wide.csv"
MANIFEST_PATH = RUN_DIR / "manifest.json"
SUMMARY_PATH = RUN_DIR / "summary.json"

EXPECTED_TOTAL = len(test_df) * len(SEEDS)

if LONG_PATH.exists():
    existing_df = pd.read_csv(LONG_PATH)
else:
    existing_df = pd.DataFrame()

completed_keys = set()

if not existing_df.empty:
    required_resume_cols = {
        "row_id",
        "candidate_index",
        "seed",
    }

    if not required_resume_cols.issubset(
        existing_df.columns
    ):
        raise RuntimeError(
            "candidates_long.csv lama tidak kompatibel untuk resume."
        )

    for r in existing_df.itertuples(index=False):
        completed_keys.add(
            (
                str(r.row_id),
                int(r.candidate_index),
                int(r.seed),
            )
        )

print("RUN_ID               :", RUN_ID)
print("RUN_DIR              :", RUN_DIR)
print("Expected candidates  :", EXPECTED_TOTAL)
print("Already completed    :", len(completed_keys))
print("Remaining            :", EXPECTED_TOTAL - len(completed_keys))

RUN_ID               : run_20260813T180837Z
RUN_DIR              : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/lora_locked/run_20260813T180837Z
Expected candidates  : 265
Already completed    : 0
Remaining            : 265


## 8. Final LoRA generation — 53 × 5

In [8]:
records = []

if LONG_PATH.exists():
    records = (
        pd.read_csv(LONG_PATH)
        .to_dict(orient="records")
    )

remaining_total = (
    len(test_df) * len(SEEDS)
    - len(completed_keys)
)

progress = tqdm(
    total=remaining_total,
    desc="LoRA FINAL 53×5",
)

for row in test_df.itertuples(index=False):

    for candidate_index, seed in enumerate(
        SEEDS,
        start=1,
    ):

        key = (
            str(row.row_id),
            int(candidate_index),
            int(seed),
        )

        if key in completed_keys:
            continue

        candidate_text, elapsed = (
            generate_one_candidate(
                duri_text=str(row.duri_text),
                seed=int(seed),
            )
        )

        record = {
            "scenario":
                "lora_locked",
            "row_id":
                str(row.row_id),
            "conversation_no":
                row.conversation_no,
            "turn_order":
                row.turn_order,
            "duri_text":
                str(row.duri_text),
            "candidate_index":
                int(candidate_index),
            "seed":
                int(seed),
            "temperature":
                float(GENERATION_CONFIG["temperature"]),
            "top_p":
                float(GENERATION_CONFIG["top_p"]),
            "top_k":
                int(GENERATION_CONFIG["top_k"]),
            "num_beams":
                int(GENERATION_CONFIG["num_beams"]),
            "max_new_tokens":
                int(GENERATION_CONFIG["max_new_tokens"]),
            "candidate_text":
                candidate_text,
            "is_empty":
                bool(not candidate_text.strip()),
            "generation_time_seconds":
                float(elapsed),
            "generated_at_utc":
                datetime.now(timezone.utc).isoformat(),
        }

        records.append(record)
        completed_keys.add(key)

        # Save setiap kandidat agar aman jika koneksi/kernel terputus.
        current_df = pd.DataFrame(records)

        current_df = current_df.sort_values(
            [
                "row_id",
                "candidate_index",
                "seed",
            ],
            kind="stable",
        )

        current_df.to_csv(
            LONG_PATH,
            index=False,
        )

        progress.update(1)

progress.close()

candidate_df = pd.read_csv(LONG_PATH)

if len(candidate_df) != EXPECTED_TOTAL:
    raise RuntimeError(
        f"Final candidates harus {EXPECTED_TOTAL}, "
        f"ditemukan {len(candidate_df)}."
    )

if candidate_df[
    [
        "row_id",
        "candidate_index",
        "seed",
    ]
].duplicated().any():
    raise RuntimeError(
        "Ada kandidat duplikat berdasarkan row_id/candidate_index/seed."
    )

counts = (
    candidate_df
    .groupby("row_id")
    .size()
)

if not (counts == len(SEEDS)).all():
    bad = counts[
        counts != len(SEEDS)
    ]
    raise RuntimeError(
        f"Tidak semua row memiliki 5 kandidat: {bad.to_dict()}"
    )

print("\nLORA FINAL GENERATION: COMPLETE 53 × 5")
print("candidates_long:", LONG_PATH)
print("Empty outputs  :", int(candidate_df["is_empty"].sum()))

LoRA FINAL 53×5:   0%|          | 0/265 [00:00<?, ?it/s]


LORA FINAL GENERATION: COMPLETE 53 × 5
candidates_long: /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/lora_locked/run_20260813T180837Z/candidates_long.csv
Empty outputs  : 0


## 9. Wide output + manifest + summary

In [9]:
wide_base = (
    candidate_df[
        [
            "row_id",
            "conversation_no",
            "turn_order",
            "duri_text",
        ]
    ]
    .drop_duplicates(
        subset=["row_id"],
        keep="first",
    )
    .copy()
)

candidate_pivot = (
    candidate_df
    .pivot(
        index="row_id",
        columns="candidate_index",
        values="candidate_text",
    )
)

candidate_pivot.columns = [
    f"candidate_{int(c)}"
    for c in candidate_pivot.columns
]

seed_pivot = (
    candidate_df
    .pivot(
        index="row_id",
        columns="candidate_index",
        values="seed",
    )
)

seed_pivot.columns = [
    f"seed_{int(c)}"
    for c in seed_pivot.columns
]

wide_df = (
    wide_base
    .set_index("row_id")
    .join(candidate_pivot)
    .join(seed_pivot)
    .reset_index()
)

wide_df.to_csv(
    WIDE_PATH,
    index=False,
)

manifest = {
    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "run_id":
        RUN_ID,
    "scenario":
        "lora_locked",
    "model_id":
        MODEL_ID,
    "adapter_dir":
        str(ADAPTER_DIR),
    "locked_config_path":
        str(CONFIG_PATH),
    "locked_checkpoint":
        cfg.get("selected_checkpoint_name"),
    "locked_checkpoint_epoch":
        cfg.get("selected_checkpoint_epoch"),
    "locked_validation_chrfpp":
        cfg.get("validation_chrfpp"),
    "locked_validation_bleu":
        cfg.get("validation_bleu"),
    "test_rows":
        len(test_df),
    "test_source_fingerprint":
        TEST_SOURCE_FINGERPRINT,
    "test_reference_read":
        False,
    "candidate_count_per_input":
        len(SEEDS),
    "total_candidates":
        len(candidate_df),
    "seeds":
        SEEDS,
    "generation":
        GENERATION_CONFIG,
    "prompt_protocol":
        PROMPT_PROTOCOL,
    "prompt_signature":
        PROMPT_SIGNATURE,
    "precision":
        "bfloat16",
    "quantization":
        False,
    "local_files_only":
        True,
    "hf_hub_cache":
        str(HF_HUB_CACHE),
    "python":
        platform.python_version(),
    "torch":
        torch.__version__,
    "gpu":
        torch.cuda.get_device_name(DEVICE_INDEX),
    "outputs": {
        "candidates_long":
            str(LONG_PATH),
        "candidates_wide":
            str(WIDE_PATH),
    },
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

summary = {
    "status":
        "complete",
    "run_id":
        RUN_ID,
    "test_rows":
        len(test_df),
    "candidates_per_input":
        len(SEEDS),
    "total_candidates":
        len(candidate_df),
    "empty_outputs":
        int(candidate_df["is_empty"].sum()),
    "mean_generation_time_seconds":
        float(
            candidate_df[
                "generation_time_seconds"
            ].mean()
        ),
    "candidates_long":
        str(LONG_PATH),
    "candidates_wide":
        str(WIDE_PATH),
    "manifest":
        str(MANIFEST_PATH),
}

SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

ACTIVE_RUN_PATH.write_text(
    json.dumps(
        {
            "run_id": RUN_ID,
            "status": "complete",
            "completed_at_utc":
                datetime.now(timezone.utc).isoformat(),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("candidates_long :", LONG_PATH)
print("candidates_wide :", WIDE_PATH)
print("manifest        :", MANIFEST_PATH)
print("summary         :", SUMMARY_PATH)

candidates_long : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/lora_locked/run_20260813T180837Z/candidates_long.csv
candidates_wide : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/lora_locked/run_20260813T180837Z/candidates_wide.csv
manifest        : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/lora_locked/run_20260813T180837Z/manifest.json
summary         : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/lora_locked/run_20260813T180837Z/summary.json


## 10. Final audit + GPU cleanup

In [10]:
final_df = pd.read_csv(LONG_PATH)

audit = {
    "test_rows":
        len(test_df) == 53,
    "total_candidates":
        len(final_df) == 265,
    "five_per_row":
        bool(
            (
                final_df
                .groupby("row_id")
                .size()
                == 5
            ).all()
        ),
    "seeds_exact":
        set(final_df["seed"].astype(int).unique())
        == set(SEEDS),
    "reference_not_loaded":
        "indonesia_text" not in test_df.columns,
    "adapter_exists":
        ADAPTER_DIR.exists(),
    "manifest_exists":
        MANIFEST_PATH.exists(),
    "summary_exists":
        SUMMARY_PATH.exists(),
}

failed = [
    name
    for name, ok in audit.items()
    if not bool(ok)
]

display(
    pd.DataFrame(
        [
            {
                "check": name,
                "ok": bool(ok),
            }
            for name, ok in audit.items()
        ]
    )
)

if failed:
    raise RuntimeError(
        f"FINAL AUDIT GAGAL: {failed}"
    )

# Lepaskan model agar VRAM kembali kosong tanpa harus menunggu shutdown kernel.
try:
    del model
except Exception:
    pass

try:
    del base_model
except Exception:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

free_b, total_b = torch.cuda.mem_get_info(DEVICE_INDEX)

print("\n" + "=" * 74)
print("LORA FINAL TEST CANDIDATE GENERATION: COMPLETE 53 × 5")
print("=" * 74)
print("Output :", LONG_PATH)
print("VRAM free after cleanup:",
      f"{free_b/(1024**3):.2f} / {total_b/(1024**3):.2f} GiB")
print("\nSelanjutnya: S3E + MBR pada lima kandidat, lalu evaluasi.")

,check,ok
0,test_rows,True
1,total_candidates,True
2,five_per_row,True
3,seeds_exact,True
4,reference_not_loaded,True
5,adapter_exists,True
6,manifest_exists,True
7,summary_exists,True



LORA FINAL TEST CANDIDATE GENERATION: COMPLETE 53 × 5
Output : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/lora_locked/run_20260813T180837Z/candidates_long.csv
VRAM free after cleanup: 22.38 / 23.52 GiB

Selanjutnya: S3E + MBR pada lima kandidat, lalu evaluasi.
